# Task 1: Sokoban Problem 

## 1. State-Space Search Formulation
To solve the Sokoban puzzle, we formulate it as a state-space search:
* **State Representation**: A tuple `(agent_pos, boxes_pos)` where `agent_pos` is the `(x, y)` coordinate of the agent and `boxes_pos` is a sorted tuple of `(x, y)` coordinates for all boxes.
* **Initial State**: The starting coordinates of the agent (parsed from `A`) and the boxes (parsed from `B` and `C`).
* **Actions**: A list of valid movements: `North`, `South`, `East`, `West`.
* **Transition Model**: Given a state and an action, the new state is determined by moving the agent. If a box is in the way, it is pushed if the space behind it is blank (` `) or a destination (`D`). Walls (`%`) block movement[cite: 1].
* **Goal Test**: The state is a goal if all box coordinates match the designated destination coordinates (`D` or `C`)[cite: 1].
* **Path Cost**: Each action (movement or push) adds a cost of 1 to the total cost[cite: 1].

In [15]:
import heapq
import sys
import time
from collections import deque


class SokobanState:
    def __init__(self, agent, boxes, cost=0, path=None):
        self.agent = agent
        self.boxes = tuple(sorted(boxes))
        self.cost = cost
        self.path = path if path is not None else []

    def __lt__(self, other):
        return self.cost < other.cost
        
    def __eq__(self, other):
        return self.agent == other.agent and self.boxes == other.boxes
        
    def __hash__(self):
        return hash((self.agent, self.boxes))

class SokobanMap:
    def __init__(self, filepath):
        self.walls = set()
        self.destinations = set()
        self.initial_agent = None
        self.initial_boxes = set()
        self.load_map(filepath)

    def load_map(self, filepath):
        # Parses %, A, B, D, C, and blank cells
        with open(filepath, 'r') as f:
            for y, line in enumerate(f.readlines()):
                for x, char in enumerate(line.strip('\n')):
                    if char == '%':
                        self.walls.add((x, y))
                    elif char == 'A':
                        self.initial_agent = (x, y)
                    elif char == 'B':
                        self.initial_boxes.add((x, y))
                    elif char == 'D':
                        self.destinations.add((x, y))
                    elif char == 'C':
                        self.initial_boxes.add((x, y))
                        self.destinations.add((x, y))
                        
    def is_goal(self, state):
        return set(state.boxes) == self.destinations

In [16]:
def precompute_bfs_heuristic(sokoban_map):
    # Precompute true grid distances from all points to destinations, ignoring boxes but respecting walls.
    distances = {}
    for dest in sokoban_map.destinations:
        queue = deque([(dest, 0)])
        visited = {dest}
        while queue:
            current, dist = queue.popleft()
            if current not in distances or dist < distances[current]:
                distances[current] = dist
            
            x, y = current
            for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                neighbor = (x + dx, y + dy)
                if neighbor not in sokoban_map.walls and neighbor not in visited:
                    visited.add(neighbor)
                    queue.append((neighbor, dist + 1))
    return distances

def heuristic(state, bfs_distances):
    # Sum of exact BFS paths from each box to the nearest unassigned destination
    return sum(bfs_distances.get(box, float('inf')) for box in state.boxes)

def get_successors(state, sokoban_map):
    successors = []
    x, y = state.agent
    directions = {'North': (0, -1), 'South': (0, 1), 'East': (1, 0), 'West': (-1, 0)} #[cite: 1]
    
    for action, (dx, dy) in directions.items():
        new_agent = (x + dx, y + dy)
        if new_agent in sokoban_map.walls:
            continue
            
        new_boxes = set(state.boxes)
        if new_agent in new_boxes:
            box_new_pos = (new_agent[0] + dx, new_agent[1] + dy)
            if box_new_pos in sokoban_map.walls or box_new_pos in new_boxes:
                continue
            new_boxes.remove(new_agent)
            new_boxes.add(box_new_pos)
            
        successors.append(SokobanState(new_agent, new_boxes, state.cost + 1, state.path + [action]))
    return successors

def a_star_search(sokoban_map):
    bfs_dist = precompute_bfs_heuristic(sokoban_map)
    start_state = SokobanState(sokoban_map.initial_agent, sokoban_map.initial_boxes)
    frontier = []
    heapq.heappush(frontier, (heuristic(start_state, bfs_dist), start_state))
    explored = set()
    
    nodes_generated = 0
    while frontier:
        _, current = heapq.heappop(frontier)
        if sokoban_map.is_goal(current):
            return current.path, current.cost, nodes_generated
            
        if current in explored: continue
        explored.add(current)
        
        for succ in get_successors(current, sokoban_map):
            if succ not in explored:
                nodes_generated += 1
                f_cost = succ.cost + heuristic(succ, bfs_dist)
                heapq.heappush(frontier, (f_cost, succ))
    return None, 0, nodes_generated

def uniform_cost_search(sokoban_map):
    # UCS is essentially A* with h(n) = 0[cite: 1]
    start_state = SokobanState(sokoban_map.initial_agent, sokoban_map.initial_boxes)
    frontier = []
    heapq.heappush(frontier, (start_state.cost, start_state))
    explored = set()
    
    nodes_generated = 0
    while frontier:
        _, current = heapq.heappop(frontier)
        if sokoban_map.is_goal(current):
            return current.path, current.cost, nodes_generated
            
        if current in explored: continue
        explored.add(current)
        
        for succ in get_successors(current, sokoban_map):
            if succ not in explored:
                nodes_generated += 1
                heapq.heappush(frontier, (succ.cost, succ))
    return None, 0, nodes_generated

## Heuristic Admissibility and Consistency
* **Admissibility**: The proposed BFS-distance heuristic never overestimates the true cost to reach the goal. It calculates the exact number of steps required to move a box to a destination ignoring the agent's movement and other box collisions. Therefore, actual cost $\ge$ estimated cost.
* **Consistency**: $h(n) \le c(n, a, n') + h(n')$. A single action costs 1. Moving a box towards a destination reduces its BFS distance by at most 1, making the heuristic difference between states at most 1.

In [17]:
def run_experiments():
    # Placeholder for 'example_map.txt'[cite: 1]
    map_env = SokobanMap('example_map.txt') 
    
    print("Running UCS...")
    start_t = time.time()
    ucs_path, ucs_cost, ucs_space = uniform_cost_search(map_env)
    ucs_time = time.time() - start_t
    
    print("Running A*...")
    start_t = time.time()
    ast_path, ast_cost, ast_space = a_star_search(map_env)
    ast_time = time.time() - start_t
    
    print(f"{'Algorithm':<10} | {'Time (s)':<10} | {'Space (Nodes)':<15} | {'Cost':<5}")
    print("-" * 50)
    print(f"{'UCS':<10} | {ucs_time:<10.4f} | {ucs_space:<15} | {ucs_cost:<5}")
    print(f"{'A*':<10} | {ast_time:<10.4f} | {ast_space:<15} | {ast_cost:<5}")

# run_experiments()

In [18]:
import pygame

def run_gui(map_env, solution_path):
    pygame.init()
    screen = pygame.display.set_mode((800, 600))
    pygame.display.set_caption("Sokoban AI")
    font = pygame.font.SysFont(None, 36)
    
    # State tracking
    current_step = 0
    paused = False
    running = True
    
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_SPACE: #[cite: 1]
                    paused = not paused
                elif event.key == pygame.K_RIGHT: #[cite: 1]
                    if current_step < len(solution_path): current_step += 1
                elif event.key == pygame.K_LEFT: #[cite: 1]
                    if current_step > 0: current_step -= 1

        screen.fill((255, 255, 255))
        
        # Display the number of actions[cite: 1]
        step_text = font.render(f"Actions taken: {current_step} / {len(solution_path)}", True, (0, 0, 0))
        screen.blit(step_text, (10, 10))
        
        # TODO: Render map_env walls, boxes, agent based on current_step state history
        
        pygame.display.flip()
        
    pygame.quit()

## Competitive Two-Agent Problem
* **State**: `(agent1_pos, agent2_pos, boxes_pos, agent1_score, agent2_score, current_step)`
* **Rules**: 
  - Agents choose actions simultaneously at each step[cite: 1].
  - Time limit for decision-making step: 1,000 ms[cite: 1].
  - Agents cannot pass through each other[cite: 1].
  - A box placed by one agent can be pushed out by the other[cite: 1].
* **Winning Condition**: After $n$ steps (input by the user), the agent that completes the most boxes wins[cite: 1].

In [19]:
# Load the map from the text file
map_filepath = 'example_map.txt'
sokoban_map = SokobanMap(map_filepath)

print(f"Agent initial position: {sokoban_map.initial_agent}")
print(f"Number of boxes: {len(sokoban_map.initial_boxes)}")
print(f"Number of destinations: {len(sokoban_map.destinations)}\n")

# Run A* Search
print("Running A* Search...")
start_time = time.time()
path, cost, nodes_generated = a_star_search(sokoban_map)
execution_time = time.time() - start_time

# Display Results
if path is not None:
    print(f"Solution found in {execution_time:.4f} seconds!")
    print(f"Total Cost (Steps): {cost}")
    print(f"Nodes Generated: {nodes_generated}")
    print(f"Actions: {path}")
else:
    print("No solution found. Check if the map is solvable or if the agent is trapped.")

Agent initial position: (2, 2)
Number of boxes: 7
Number of destinations: 7

Running A* Search...


KeyboardInterrupt: 